In [1]:
import json
from typing import List
from processing.en import ConformerProcessor

kenlm python bindings are not installed. Most likely you want to install it using: pip install https://github.com/kpu/kenlm/archive/master.zip
kenlm python bindings are not installed. Most likely you want to install it using: pip install https://github.com/kpu/kenlm/archive/master.zip


In [2]:
processor = ConformerProcessor('./vocabulary/en_graph.json')

In [34]:
data = json.load(open('./vocabulary/en_graph.json', 'r'))

In [4]:
data.keys()

dict_keys(['vowel', 'consonant', 'composed_vowel', 'composed_consonant', 'mixed', 'last_item', 'last_special', 'last_split_item'])

In [5]:
def get_last_item(word: str, pattern: str):
    length_item = len(pattern)
    start_check = len(word) - length_item
    if word[start_check: ] == pattern:
        return word[:start_check ], pattern
    return None

In [6]:
def get_last_pattern(word: str, patterns: List[str]):
    item = None
    for pattern in patterns:
        items = get_last_item(word, pattern)
        if items is not None:
            word = items[0]
            item = items[1]
            break

    if item is None:
        return None

    length = None
    arr_pattern = None
    best_pattern = None

    for pattern in patterns[item]:
        if pattern not in word:
            continue
        
        arr = word.split(pattern)
        if length is None or (length > len(arr[-1])):
            length = len(arr[-1])
            arr_pattern = arr
            best_pattern = pattern
    
    if arr_pattern is None or len(arr_pattern) == 1:
        return None
    
    return best_pattern.join(arr_pattern[:-1]), f"{best_pattern}_{item}", arr_pattern[-1]


In [7]:
def get_last_list(word: str, patterns: List[str]):
    for pattern in patterns:
        items = get_last_item(word, pattern)
        if items is not None:
            return items
    return None

In [8]:
def get_last_split_item(word: str, patterns: dict):
    for pattern in patterns.keys():
        items = get_last_item(word, pattern)
        if items is not None:
            pattern_items = get_last_list(items[0], list(patterns[pattern]))
            if pattern_items is not None:
                return items[0], [*items[1]]
    return None

In [9]:
def get_last_auto_split(word, patterns: dict):
    print(patterns)
    for pattern in patterns.keys():
        items = get_last_item(word, pattern)
        if items is not None:
            return items[0], patterns[pattern]
    return None

In [10]:
def get_first_item(word: str, patterns: List[str]):
    for item in patterns:
        if item in word:
            first_item = word[:len(item)]
            if first_item == item:
                return first_item, word[len(item):]
    return None

In [11]:
def word2graphemes(text: str, patterns: List, n_grams: int = 4):
        if len(text) == 1:
            if text in patterns:
                return [text]
            return ["<unk>"]
        graphemes = []
        start = 0
        if len(text) - 1 < n_grams:
            n_grams = len(text)
        num_steps = n_grams
        while start < len(text):
            found = True
            item = text[start:start + num_steps]
            
            if item in patterns:
                graphemes.append(item)
            elif num_steps == 1:
                graphemes.append("<unk>")
            else:
                found = False

            if found:
                start += num_steps
                if len(text[start:]) < n_grams:
                    num_steps = len(text[start:])
                else:
                    num_steps = n_grams
            else:
                num_steps -= 1

        return graphemes

In [12]:
first_patterns = data['vowel'] + data['consonant'] + data['composed_vowel'] + data['composed_consonant']
first_patterns = sorted(first_patterns, key=len, reverse=True)

stride_patterns = first_patterns + data['mixed']

In [38]:
word = input()
first_item = ''
left_side = ''
middle_side = ''
right_side = ''
last_items = []

last_item_outputs = get_last_split_item(word, data['last_split_item'])

if last_item_outputs is not None:
    left_side = last_item_outputs[0]
    last_items += last_item_outputs[1]
    word = left_side

last_item_outputs = get_last_pattern(word, data['last_item'])
if last_item_outputs is not None:
    left_side = last_item_outputs[0]
    middle_side = last_item_outputs[1]
    right_side = last_item_outputs[2]
else:
    left_side = word
    tmp = []
    while True:
        last_item_outputs = get_last_list(left_side, data['last_special'])
        if last_item_outputs is not None:
            left_side = last_item_outputs[0]
            tmp.append(last_item_outputs[1])
        else:
            break
    tmp = [tmp[i] for i in range(len(tmp) - 1, -1, -1)]
    last_items += tmp

# First Handle
first_items = get_first_item(left_side, first_patterns)
if first_items is not None:
    first_item = first_items[0]
    left_side = first_items[1]

left_graphemes = word2graphemes(left_side, stride_patterns)
right_graphemes = word2graphemes(right_side, stride_patterns)
middle_grapheme = [middle_side] if middle_side != '' else []
first_grapheme = [first_item] if first_item != '' else []

graphemes = first_grapheme + left_graphemes + middle_grapheme + right_graphemes + last_items
print(graphemes)

['s', 'o', 'ci', 'al']


In [37]:
a = ['d']

In [191]:
a + []

['d']